<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/Fine_tune_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended runtime type: GPU

In [ ]:
try:
    import google.colab
    google.colab.drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import datasets
import datetime
import numpy
import os
import transformers
import sklearn
import sys

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

In [ ]:
from innoprod.path_tools import find_highest_numbered_subdir
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.training_run_manager import TrainingRunManager
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

# Retrieving and preparing data

In [ ]:
data = get_sheet_dfs()
roadmaps_df = wrangle_roadmaps(data['Roadmaps'])

In [ ]:
drs_col_name = 'Current Digital Readiness Score (refer to PAS:1040)'

qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

In [ ]:
drs_labels = [
    "The business has no vision for driving growth with digital technologies, and is not supporting workers to investigate opportunities.", # 1
    "Workers are aware of the potential for digital technologies and are supported by the business to experiment with local trials.", # 2
    "The business is learning from local trials of digital technologies and leaders are investigating the business case.", # 3
    "The business has a strategic vision for digital transformation, the business case is agreed and implementation is underway.", # 4
    "Workers are engaged in digital transformation and the business is starting to achieve business case benefits.", # 5
    "Digital technologies are driving continuous improvement in key aspects of operational performance including supply chain and customer services.", # 6
    "Innovation with digital technologies is part of the culture of the business and is driving continuous improvement in all aspects.", # 7
    "Increasing adoption of digital technologies is sustained by reinvestment of related profits and continuous renewal of the business case.", # 8
    "Digital technologies are driving optimized productivity and competitiveness for the business and its partners.", # 9
]

In [ ]:
qual_df = roadmaps_df[[drs_col_name] + qual_cols].copy()
qual_df = roadmaps_df.melt(id_vars=[drs_col_name], value_vars=qual_cols, var_name='Question', value_name='Response')
qual_df = qual_df.dropna()
# qual_df


In [ ]:
qual_df[drs_col_name].isna().value_counts()

In [ ]:
qual_df['labels'] = qual_df[drs_col_name].apply(lambda drs: drs -1)
qual_df['text'] = qual_df.apply(lambda row: row["Question"] + ". " + row["Response"], axis=1)
qual_df = qual_df.drop([drs_col_name, "Question", "Response"], axis=1)
qual_df

# Preparing the BERT model

In [ ]:
model_name = "bert-base-uncased"

In [ ]:
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)

We would ideally like to set a `max_length` of 1,024 but the `bert-base-uncased` model only supports a maximum sequence length of 512.

**TO DO** check for alternatives, and/or use chunking strategy with agglomeration of results.

In [ ]:
truncation = True
max_length = 512
padding="max_length"

def tokenize(batch):
  return tokenizer(
    batch['text'],
    truncation=truncation,
    max_length=max_length,
    padding=padding
  )

In [ ]:
dataset = datasets.Dataset.from_pandas(qual_df, preserve_index=False)
dataset

In [ ]:
train, test = sklearn.model_selection.train_test_split(
    qual_df, test_size=0.25, stratify=qual_df['labels'], random_state=42
)

In [ ]:
# train['labels'].value_counts().sort_index().plot(kind="bar")

In [ ]:
# test['labels'].value_counts().sort_index().plot(kind="bar")

In [ ]:
train = datasets.Dataset.from_pandas(train[['labels', 'text']], preserve_index=False)
test = datasets.Dataset.from_pandas(test[['labels', 'text']], preserve_index=False)
dataset = datasets.DatasetDict({
    'train': train,
    'test': test
})
# dataset

In [ ]:
dataset = dataset.map(lambda x: tokenize(x))
# dataset

In [ ]:
data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
# data_collator

Added `is_decoder=True` due to information message recommending this "If you want to use `BertLMHeadModel` as a standalone"

In [ ]:
config = transformers.AutoConfig.from_pretrained(
    model_name,
    num_labels=len(drs_labels),
    id2label={label: i for i, label in enumerate(drs_labels)},
    label2id={i: label for i, label in enumerate(drs_labels)},
    is_decoder=True
)

model = transformers.AutoModelForSequenceClassification.from_pretrained(
    model_name,
    config=config
  )

In [ ]:
output_dir = '/content/drive/MyDrive/Colab Notebooks/model_training/'
model_task_name = model_name + "_drs-classification"
training_run_manager = TrainingRunManager(output_dir, model_task_name)
checkpoint_dir = training_run_manager.get_highest_checkpoint_dir()
run_dir = training_run_manager.get_next_run_dir()

In [ ]:
training_args = transformers.TrainingArguments(
    output_dir=run_dir,
    resume_from_checkpoint=checkpoint_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=False, # Changed from True to False to resolve bf16/gpu compatibility error
    learning_rate=2e-5,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    hub_private_repo=True,
)
# training_args

In [ ]:
data_collator_for_classification = transformers.DataCollatorWithPadding(tokenizer=tokenizer)

trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator_for_classification
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
predictions = trainer.predict(dataset["test"])
# predictions

In [ ]:
predicted_labels = predictions.predictions.argmax(axis=-1)
# predicted_labels

The `predicted_labels` are token IDs predicted by the language model. To convert them to human-readable text, you can use the tokenizer's `decode` method. The `label_ids` from the `predictions` object represent the true token IDs from the input sequence.

Keep in mind that for a Causal Language Model, `predicted_labels` represent the most likely next token at each position, not a single classification label like a DRS score for the entire input text.

In [ ]:
# Decode a sample predicted sequence (predicted token IDs)
sample_predicted_tokens = predicted_labels[0]
decoded_predicted_text = tokenizer.decode(sample_predicted_tokens, skip_special_tokens=True)
# print(decoded_predicted_text)

In [ ]:
# Decode a sample true sequence (from label_ids in predictions, which are the target tokens for CausalLM)
sample_true_tokens = predictions.label_ids[0]
decoded_true_text = tokenizer.decode(sample_true_tokens, skip_special_tokens=True)
# print(f"Sample Decoded True Text: {decoded_true_text}")